Author: **Dongyuan Gao**

Course: HSLU Computer Vision — Lecture 3 Project

Based on the style of the lecturer's notebooks by *Safouane El Ghazouali* (TOELT LLC / HSLU).

# -----  -----  -----  -----  -----  -----  -----  -----

# 🚗 Fine-Tuning YOLOv10 for Self-Driving Object Detection

In this notebook we fine-tune a pre-trained **YOLOv10n** on the **Udacity Self-Driving Car dataset** (≈15,000 dashcam images, 11 classes: car, truck, pedestrian, biker, traffic light, traffic sign, …).

The goal: build a **real-time detector** that finds vehicles, pedestrians and traffic signals in dashcam footage — the core perception step of any self-driving stack.

### Why YOLO?
- **Real-time**: a single forward pass gives all boxes + labels.
- **Detection, not classification**: tells us *what* AND *where*.
- **Easy fine-tuning** via the Ultralytics library.

### What You'll Learn
- Downloading a pre-labelled detection dataset from Roboflow.
- Fine-tuning YOLOv10n on custom classes.
- Reading training curves and validation metrics (mAP).
- Running inference on images and videos.
- Plugging the fine-tuned weights into a **live webcam** demo on your Mac.

# 🧭 Running on DGX via VS Code Remote

Project directory on DGX: `/home/dongyuan/Desktop/computer_vision`

Typical flow:
- Connect to the DGX with VS Code Remote - SSH.
- Open this notebook **on the remote machine** (so paths refer to DGX storage).
- Use a conda env or venv with PyTorch + CUDA already installed.
- Keep datasets on DGX local storage (faster than network mounts).

# 🧰 Environment Setup (DGX)

Install Ultralytics (YOLO), Roboflow (dataset download), and OpenCV.

On a DGX, you typically already have a CUDA-enabled PyTorch in your conda env.
If you do not, create or activate your environment before running the install below.

In [9]:
!pip install -q ultralytics roboflow opencv-python
!pip install open-clip-torch
!pip install torch

### Optional: Ollama Python Client (local VLM captions)

If you want to run the VLM overlay cell later, install the **Python client** in your environment.
The Ollama server itself is installed and run in the terminal (system-level).

Example install (terminal or notebook cell): `pip install ollama`

### Import Libraries & Check GPU

On the DGX you should see `cuda` and at least one visible GPU.
If it prints `cpu`, your environment is missing CUDA-enabled PyTorch or no GPU is visible.

In [10]:
from ultralytics import YOLO
from roboflow import Roboflow
import torch
import os, glob, yaml
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn as nn
import open_clip
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

# Quick GPU visibility check on DGX
!nvidia-smi -L

# Explanation
# - device: tells YOLO where to run (GPU is ~30x faster than CPU).
# - Ultralytics auto-uses this device unless we override it.

Using device: cuda
PyTorch version: 2.11.0+cu130
GPU 0: NVIDIA GB10 (UUID: GPU-0b6645ac-fb60-3d81-c925-eb574014af92)


# 📂 Dataset on the DGX (Roboflow or Local Path)

You can either download with Roboflow **on the DGX** or point to a dataset that is already on DGX storage.

**Option A (Roboflow download on DGX):**
1. Go to https://public.roboflow.com/object-detection/self-driving-car
2. Click **Download Dataset** → pick **YOLOv8** format (compatible with v10).
3. Roboflow shows you a **personalized snippet** with your API key — paste it in the next cell.

**Option B (Dataset already on DGX):**
- Set the `DATASET_DIR` path below to the folder that contains `data.yaml`, `train/`, `valid/`, `test/`.

**Note (local path):** If you set `USE_ROBOFLOW = False`, this notebook looks for the dataset in `./Self-Driving-Car-3` or `./self-driving-car`. You can also override with an environment variable, e.g. `export DATASET_DIR=/path/to/dataset`.


In [3]:
# Set this to False if the dataset is already on DGX storage
USE_ROBOFLOW = False

# If USE_ROBOFLOW is False, set the local dataset folder on DGX
def resolve_dataset_dir() -> str:
    env_path = os.getenv("DATASET_DIR")
    if env_path:
        return env_path
    candidates = [
        os.path.join(os.getcwd(), "Self-Driving-Car-3"),
        os.path.join(os.getcwd(), "self-driving-car"),
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    raise FileNotFoundError(
        "Dataset folder not found. Set DATASET_DIR or place dataset at ./Self-Driving-Car-3 or ./self-driving-car"
    )

if USE_ROBOFLOW:
    # ---- PASTE YOUR ROBOFLOW SNIPPET HERE ----
    rf = Roboflow(api_key="YOUR_API_KEY")
    project = rf.workspace("roboflow-gw7yv").project("self-driving-car")
    dataset = project.version(3).download("yolov8")
    dataset_location = dataset.location
else:
    DATASET_DIR = resolve_dataset_dir()
    dataset_location = DATASET_DIR

data_yaml = os.path.join(dataset_location, "data.yaml")
print(f"Dataset location: {dataset_location}")
print(f"data.yaml: {data_yaml}")

# Explanation
# - dataset_location: absolute path to the dataset folder on DGX
# - data.yaml lists class names and the train/valid/test paths YOLO needs

Dataset location: /home/dongyuan/Desktop/computer_vision/Self-Driving-Car-3
data.yaml: /home/dongyuan/Desktop/computer_vision/Self-Driving-Car-3/data.yaml


## Load CLIP model and linear probe

This runtime notebook supports two modes:

- current local repo layout (`weights/clip/linear_probe`, `weights/yolo`, `original_videos`, `runs_output`),
- older or alternate layouts via environment variables or fallback path detection.

Optional environment overrides:

- `PROBE_DIR` for the CLIP linear probe directory,
- `YOLO_WEIGHTS` for the YOLO weight file,
- `INPUT_VIDEO` for the input video path,
- `OUTPUT_DIR` for the output video directory.

In [11]:
# ============================================================
# Load CLIP model for car brand classification
# ============================================================

import open_clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED,
    device=DEVICE
)

clip_model.eval()

# Load your trained linear probe
# Example: sklearn LogisticRegression / LinearSVC / etc.
# linear_probe = joblib.load("car_brand_linear_probe.pkl")
# ============================================================
# Load CLIP model + PyTorch linear probe
# ============================================================

import json
from pathlib import Path
import torch.nn as nn
import open_clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROBE_DIR = Path("weights/clip/linear_probe")

# ------------------------------------------------------------
# Load config
# ------------------------------------------------------------

with open(PROBE_DIR / "config.json", "r") as f:
    config = json.load(f)

MODEL_NAME = config["clip_model"]
PRETRAINED = config["pretrained"]
embed_dim = config["embed_dim"]
n_classes = config["n_classes"]

# ------------------------------------------------------------
# Load class names
# ------------------------------------------------------------

with open(PROBE_DIR / "class_names.json", "r") as f:
    class_names = json.load(f)

print("Classes:", class_names)

# ------------------------------------------------------------
# Load CLIP model
# ------------------------------------------------------------

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED,
    device=DEVICE
)

clip_model.eval()

# ------------------------------------------------------------
# Rebuild linear probe architecture
# ------------------------------------------------------------

linear_probe = nn.Linear(embed_dim, n_classes)

# ------------------------------------------------------------
# Load trained weights
# ------------------------------------------------------------

state_dict = torch.load(
    PROBE_DIR / "linear_probe_weights.pt",
    map_location=DEVICE
)

linear_probe.load_state_dict(state_dict)

linear_probe.to(DEVICE)
linear_probe.eval()

print("CLIP + linear probe loaded")

Classes: ['Audi', 'BMW', 'Chevrolet', 'Citroen', 'Dacia', 'Fiat', 'Ford', 'Honda', 'Hyundai', 'Kia', 'Mercedes', 'Nissan', 'Opel', 'Peugeot', 'Renault', 'Seat', 'Skoda', 'Tofaş', 'Toyota', 'Volkswagen']
CLIP + linear probe loaded


In [12]:
# ============================================================
# Predict car brand from cropped image
# ============================================================

import torch.nn.functional as F

def predict_car_brand(crop_bgr):

    # OpenCV BGR -> RGB
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    # Convert to PIL
    pil_image = Image.fromarray(crop_rgb)

    # CLIP preprocessing
    image_tensor = clip_preprocess(pil_image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        # ----------------------------------------------------
        # Image embedding
        # ----------------------------------------------------

        features = clip_model.encode_image(image_tensor)

        # SAME normalization as training
        features = F.normalize(features, dim=-1)

        # ----------------------------------------------------
        # Linear probe prediction
        # ----------------------------------------------------

        logits = linear_probe(features)

        probs = torch.softmax(logits, dim=1)

        confidence, pred_idx = probs.max(dim=1)

        confidence = confidence.item()
        pred_idx = pred_idx.item()

    brand_name = class_names[pred_idx]

    return brand_name, confidence

## Load yolo fine-tuned model

In [13]:
model = YOLO('weights/yolo/best.pt')

# 🎥 Part 2 — Video Demo (DGX Path Input)

Place a dashcam clip on the DGX (scp it from your Mac if needed).
The code below processes every frame and **saves an annotated output video** on the DGX.

In [14]:
# ============================================================
# YOLO + CLIP Car Brand Recognition on Video
# ============================================================

import cv2
import os
from pathlib import Path
from tqdm import tqdm

# ------------------------------------------------------------
# Input video
# ------------------------------------------------------------

video_path = "original_videos/dashcam.mp4"

assert os.path.exists(video_path), "Video path not found"

# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------

output_dir = Path("runs_output/detect/clip_predict")
output_dir.mkdir(parents=True, exist_ok=True)

output_video_path = output_dir / "annotated_video.mp4"

# ------------------------------------------------------------
# Open video
# ------------------------------------------------------------

cap = cv2.VideoCapture(video_path)

assert cap.isOpened(), "Could not open video"

# Video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"FPS: {fps}")
print(f"Resolution: {width}x{height}")
print(f"Frames: {frame_count}")

# ------------------------------------------------------------
# Video writer
# ------------------------------------------------------------

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(output_video_path),
    fourcc,
    fps,
    (width, height)
)

# ------------------------------------------------------------
# Process video frame-by-frame
# ------------------------------------------------------------

for _ in tqdm(range(frame_count)):

    ret, frame = cap.read()

    if not ret:
        break

    # --------------------------------------------------------
    # YOLO inference
    # --------------------------------------------------------

    results = model(frame, conf=0.4, device=DEVICE)

    result = results[0]

    names = result.names

    # --------------------------------------------------------
    # Iterate detections
    # --------------------------------------------------------

    for box in result.boxes:

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        conf = float(box.conf[0])

        cls_id = int(box.cls[0])

        class_name = names[cls_id]

        label = class_name

        # ====================================================
        # If detected object is a car -> run CLIP
        # ====================================================

        if class_name.lower() == "car":

            # Optional size filtering
            if (x2 - x1) > 80 and (y2 - y1) > 80:

                # Crop car
                car_crop = frame[y1:y2, x1:x2]

                if car_crop.size > 0:

                    try:

                        brand, brand_conf = predict_car_brand(car_crop)

                        label = f"{brand} ({brand_conf:.2f})"

                    except Exception as e:

                        print(f"CLIP error: {e}")

        # ----------------------------------------------------
        # Draw bounding box
        # ----------------------------------------------------

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # ----------------------------------------------------
        # Draw label
        # ----------------------------------------------------

        cv2.putText(
            frame,
            f"{label} {conf:.2f}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    # --------------------------------------------------------
    # Write frame
    # --------------------------------------------------------

    writer.write(frame)

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

cap.release()
writer.release()

print(f"Saved annotated video to:")
print(output_video_path)

FPS: 30
Resolution: 1920x1080
Frames: 450


  0%|          | 0/450 [00:00<?, ?it/s]


0: 288x512 1 car, 3.1ms
Speed: 0.9ms preprocess, 3.1ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)


  0%|          | 1/450 [00:00<00:47,  9.46it/s]


0: 288x512 1 car, 3.2ms
Speed: 0.7ms preprocess, 3.2ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 1 car, 3.3ms
Speed: 1.0ms preprocess, 3.3ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 1 car, 3.2ms
Speed: 0.6ms preprocess, 3.2ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 1 car, 3.3ms
Speed: 0.7ms preprocess, 3.3ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)


...


0: 288x512 6 cars, 3.0ms
Speed: 0.6ms preprocess, 3.0ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 3 cars, 3.0ms
Speed: 0.6ms preprocess, 3.0ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 6 cars, 3.1ms
Speed: 0.6ms preprocess, 3.1ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 5 cars, 3.1ms
Speed: 0.6ms preprocess, 3.1ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)


 99%|█████████▉| 446/450 [00:13<00:00, 34.55it/s]


0: 288x512 6 cars, 3.1ms
Speed: 0.6ms preprocess, 3.1ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 5 cars, 3.4ms
Speed: 0.7ms preprocess, 3.4ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 5 cars, 3.2ms
Speed: 0.7ms preprocess, 3.2ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 5 cars, 3.2ms
Speed: 0.6ms preprocess, 3.2ms inference, 0.2ms postprocess per image at shape (1, 3, 288, 512)


100%|██████████| 450/450 [00:13<00:00, 32.16it/s]

Saved annotated video to:
runs_output/detect/clip_predict/annotated_video.mp4
